# System RAG dla teledetekcji — demo

Notebook pokazuje działanie systemu RAG zbudowanego od podstaw (bez LangChain),
opartego o **FAISS** i model **Gemini** (generacja + embeddingi). Korpus to kilka
publikacji naukowych o klasyfikacji pokrycia terenu z obrazów satelitarnych
(pliki PDF w katalogu `dane/`).

In [ ]:
# !pip install -q google-genai faiss-cpu pypdf numpy python-dotenv

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from rdzen import SilnikRAG

## 1. Budowa bazy wektorowej

Pipeline: wczytanie PDF → podział na fragmenty zdaniowe → embeddingi Gemini →
indeks FAISS. Wynik zapisywany jest na dysk (`indeks/`), więc kolejne
uruchomienia mogą tylko go wczytać.

> Uwaga: darmowy plan API ogranicza liczbę embeddingów do ~100/min, więc dla
> większego korpusu krok ten celowo zwalnia (rate limiting).

In [ ]:
silnik = SilnikRAG()
# Zbuduj indeks od zera (wolniejsze) ...
# liczba = silnik.zbuduj_indeks()
# ... albo wczytaj gotowy z dysku:
silnik.wczytaj_indeks()
print('Liczba fragmentów w bazie:', silnik.baza.indeks.ntotal)

## 2. Zapytania w domenie

System wyszukuje najbardziej podobne fragmenty i generuje odpowiedź tylko na ich
podstawie, podając źródła (plik + strona) oraz podobieństwo kosinusowe.

In [ ]:
def pokaz(pytanie):
    w = silnik.zapytaj(pytanie)
    print('PYTANIE:', pytanie)
    print('\nODPOWIEDŹ:', w.odpowiedz)
    print('\nŹRÓDŁA:')
    for i, (fr, sim) in enumerate(w.zrodla, 1):
        print(f'  [{i}] {fr.zrodlo}, str. {fr.strona}  (podobieństwo {sim:.3f})')

pokaz('Jakie architektury sieci neuronowych stosuje się do klasyfikacji pokrycia terenu?')

In [ ]:
pokaz('Jakie jest znaczenie danych wieloczasowych (multi-temporal) w klasyfikacji pokrycia terenu?')

## 3. Pytanie spoza korpusu

Gdy w dokumentach nie ma odpowiedzi, system nie zmyśla — odmawia. Zwróćmy uwagę
na wyraźnie niższe podobieństwo znalezionych fragmentów.

In [ ]:
pokaz('Jaka jest stolica Francji?')